In [ ]:
import numpy as np
import json
import pandas as pd
# from tensorflow.keras.metrics import MeanSquaredError
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv3D
from sklearn.model_selection import KFold 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from scipy.stats import kendalltau
import seaborn as sns
from tqdm import tqdm
import os

In [ ]:
lookback = 1
forecast_horizon = 1
set_lookback = 1

In [ ]:
#File paths for running with real data

#Where the entire dataset split into its timestamps is stored
timestamps_directory = 'split_files_cleanedVEG/'

#Where the list of all timestamps are stored
timestamps_file_path = os.path.join(timestamps_directory, 'alltimestamps_cleanedVEG.json')

#Where the individual samples are stored
saved_files = 'lookback_and_lookahead_files_cleanedVEG/'

#Where the list of timestamps in their splits are stored
split_file = 'timestamps_splits_cleanedVEG.npz'

In [ ]:
# Utility to load all timestamps
def load_all_timestamps():
    with open(timestamps_file_path, 'r') as file:
        timestamps = json.load(file)
        sorted_timestamps = sorted(timestamps)
        return sorted_timestamps

# Utilities to find lengths
def load_split_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    return train_len, train_len + val_len 

def get_all_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    test_len = len(loaded_data['test'])
    return train_len, val_len, test_len

# Function for loading a single training sample
def load_singular_train_data(index, lookback):
    all_timestamps = load_all_timestamps()
    timestamp_name = all_timestamps[lookback + index]
    file_name = f'{index + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

# Function for loading a single validation sample
def load_singular_val_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, _ = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_len + index]
    file_name = f'{index + train_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

# Function for loading a single test sample
def load_singular_test_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, train_val_len = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_val_len + index]
    file_name = f'{index + train_val_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    try:
        data = np.load(file_path, allow_pickle=True)
    except:
        print("The last erroneous files don't exist")

    return data['X_batches'], data['y_batches']

In [ ]:
def fill_none_with_mean(values):
    array = np.array([np.nan if v is None else v for v in values])
    nan_indices = np.isnan(array)
    non_nan_indices = np.where(~nan_indices)[0]
    non_nan_values = array[non_nan_indices]
    if len(non_nan_values) == 0: 
        return np.zeros_like(array).tolist()
    array[nan_indices] = np.interp(np.where(nan_indices)[0], non_nan_indices, non_nan_values)
    return array.tolist()



In [ ]:
#Implementation of the TimeSeriesDataset

from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, indices, lookback, mode="train"):
        """
        The parameters are: 
        indices is a list of indices, 
        lookback is set manually, 
        mode to indicate how we are appropriately adding the indices.
        
        """
        self.indices = indices
        self.lookback = lookback
        self.mode = mode
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        index = self.indices[idx]

        # Load data based on mode
        if self.mode == "train":
            X, y = load_singular_train_data(index, self.lookback)
        elif self.mode == "val":
            X, y = load_singular_val_data(index, self.lookback)
        elif self.mode == "test":
            X, y = load_singular_test_data(index, self.lookback)

        for i, array in enumerate(X):
            for j, sub_array in enumerate(array):
                X[i][j] = fill_none_with_mean(sub_array)
        
        X = np.array(X)  

        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        
        return X_tensor, y_tensor


In [ ]:
#Getting lengths of the training, validation and testing
train_length, val_length, test_length = get_all_lengths()

In [ ]:
# Split data indices for train, validation, and test
train_length, val_length, test_length = get_all_lengths()

#From 0 up until the length of the training, validation and test sets 

#We really should check this with training data
train_indices = list(range(0, train_length))
val_indices = list(range(0, val_length))
test_indices = list(range(0, test_length))

# Initialize Datasets
train_dataset = TimeSeriesDataset(train_indices, lookback, mode="train")
val_dataset = TimeSeriesDataset(val_indices, lookback, mode="val")
test_dataset = TimeSeriesDataset(test_indices, lookback, mode="test")

# Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
#Normalization functions

# Load global statistics from JSON file
import json

with open('globaldatastatistics_withVEG.json', 'r') as f:
    global_stats = json.load(f)

# Function to normalize data using global statistics
def normalize_global(batch, variable_name):
    mean = global_stats[variable_name]['mean']
    std = global_stats[variable_name]['std']
    return (batch - mean) / (std)

# Function to normalize data using local statistics
def normalize_local(batch):
    mean = batch.mean()  # Compute mean across the time dimension (axis=1)
    std = batch.std()# Compute std deviation across time dimension
    return (batch - mean) / (std)    # Normalize batch and leave a small epsilon to avoid division by 0

def normalize_precipitation(batch):
    log_normalized = torch.log(batch + 1)
    zero_indicator = (batch == 0).float()
    return log_normalized, zero_indicator


In [ ]:
#Variable names
variable_names = ['10 metre U wind component', '10 metre V wind component', '2 metre dewpoint temperature', '2 metre temperature', 'UV visible albedo for direct radiation (climatological)', 'Total column rain water', 'Volumetric soil water layer 1', 'Leaf area index, high vegetation', 'Leaf area index, low vegetation', 'Forecast surface roughness', 'Total precipitation', 'Time-integrated surface latent heat net flux', 'Evaporation']

In [ ]:
#MLP definition
class MLP_5D(nn.Module):
    def __init__(self, height, width):
        super(MLP_5D, self).__init__()
        # Define the fully connected layers
        self.fc1 = nn.Linear(8, 128)  # Input channels = 41, output features = 128
        self.dropout1 = nn.Dropout(0.05)
        self.fc2 = nn.Linear(128, 64)  # Output features = 64
        self.dropout2 = nn.Dropout(0.05)
        self.fc3 = nn.Linear(64, 1)    # Final output, reducing to 1 channel

        self.height = height
        self.width = width

    def forward(self, x):
        batch_size, timesteps, channels, height, width = x.shape
        
        # Ensure the input spatial dimensions match the expected height and width
        assert height == self.height and width == self.width, "Height and width mismatch"
        
        # Reshape to (batch * timesteps * height * width, channels)
        x = x.permute(0, 1, 3, 4, 2).reshape(-1, channels)
        # print(x.shape)
        
        # Apply MLP
        x = self.fc1(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = torch.nn.functional.softplus(x)
        
        # Reshape back to (batch, timesteps, 1, height, width)
        x = x.view(batch_size, timesteps, self.height, self.width, 1).permute(0, 1, 4, 2, 3)

        return x

In [ ]:
#MLP definition
class MLP_5D2(nn.Module):
    def __init__(self, height, width):
        super(MLP_5D2, self).__init__()
        # Define the fully connected layers
        self.fc1 = nn.Linear(16, 128)  # Input channels = 41, output features = 128
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 64)  # Output features = 64
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(64, 1)    # Final output, reducing to 1 channel

        self.height = height
        self.width = width

    def forward(self, x):
        batch_size, timesteps, channels, height, width = x.shape
        
        # Ensure the input spatial dimensions match the expected height and width
        assert height == self.height and width == self.width, "Height and width mismatch"
        
        # Reshape to (batch * timesteps * height * width, channels)
        x = x.permute(0, 1, 3, 4, 2).reshape(-1, channels)
        
        # Apply MLP (Fully connected layers)
        x = self.fc1(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = torch.nn.functional.softplus(x)
        
        # Reshape back to (batch, timesteps, 1, height, width)
        x = x.view(batch_size, timesteps, self.height, self.width, 1).permute(0, 1, 4, 2, 3)

        return x

In [ ]:
#ConvLSTM definition
from ConvLSTM import ConvLSTM
import torch
import torch.nn as nn
from collections import defaultdict

class ConvLSTMNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dims, kernel_size, num_layers, output_channels, batch_first=True, pool_size=(2,2)):
        super(ConvLSTMNetwork, self).__init__()
        
        # ConvLSTM module
        self.convlstm = ConvLSTM(input_dim=input_dim,
                                 hidden_dim=hidden_dims,
                                 kernel_size=kernel_size,
                                 num_layers=num_layers,
                                 batch_first=batch_first,
                                 bias=True,
                                 return_all_layers=True)
        
        # Batch Normalization for each ConvLSTM layer's output
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm3d(hidden_dim) for hidden_dim in hidden_dims
        ])

        # Final Conv3D layer for regression pathway
        self.conv3d = nn.Conv3d(in_channels=hidden_dims[-1],
                                out_channels=output_channels,
                                kernel_size=(1, 3, 3),
                                padding=(0, 1, 1))

        # MLP for regression output: (B,T,C,H,W) -> (B,T,1,H,W)
        self.mlp = MLP_5D(height=81, width=97)

        self.classification_head = nn.Sequential(
            nn.Conv3d(output_channels, 1, kernel_size=(1,1,1)),  # from C to 1 channel
            nn.Sigmoid()
        )

        self.activation_variance = defaultdict(list)

    def forward(self, x):
        """
        x: (B, T, input_dim, H, W)
        """
        # Forward through ConvLSTM
        layer_output_list, last_state_list = self.convlstm(x)
        
        # Apply batch norms
        for i, output in enumerate(layer_output_list):
            # output: (B, T, C, H, W)
            output = output.permute(0, 2, 1, 3, 4)  # (B, C, T, H, W) for BatchNorm3d
            output = self.batch_norms[i](output)
            output = output.permute(0, 2, 1, 3, 4)  # back to (B, T, C, H, W)

            #Track variance across spatial dimensions for hooks with activation tracking 
            activation_variance = output.var(dim=(3, 4)).mean().item()
            self.activation_variance[f"ConvLSTM_layer_{i}"].append(activation_variance)

            layer_output_list[i] = output
        
        # Take output from the last ConvLSTM layer
        final_output = layer_output_list[-1]  # (B, T, C, H, W)

        # Pass through Conv3D: needs (B,C,T,H,W)
        final_output = final_output.permute(0, 2, 1, 3, 4)  # (B,C,T,H,W)
        final_output = self.conv3d(final_output)
        # Now final_output: (B, output_channels, T, H, W)

        # Return to (B,T,C,H,W) for MLP (regression)
        final_output_t = final_output.permute(0, 2, 1, 3, 4)  # (B,T,C,H,W)

        # Regression output
        regression_output = self.mlp(final_output_t)  # (B,T,1,H,W)

        # Classification output:
        # The classification head is defined for (B,C,T,H,W), so reorder again
        final_output_c = final_output  # still (B,output_channels,T,H,W)
        classification_output = self.classification_head(final_output_c)
        # classification_output: (B,1,T,H,W)

        # Permute classification output to match (B,T,1,H,W) format
        classification_output = classification_output.permute(0, 2, 1, 3, 4)  # (B,T,1,H,W)

        return regression_output, classification_output, final_output_t

In [ ]:
# Gating Mechanism: Learnable gate to control the flow of vegetation features
class GatingMechanism(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(GatingMechanism, self).__init__()
        # A learnable gate for each feature (vegetation feature)
        self.gate = nn.Sequential(
            nn.Linear(81*97*16, hidden_dim),  # Hidden layer to transform the features
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, 1),          # Output a gate value for each feature (sigmoid activation)
            nn.Sigmoid()                    # Sigmoid to constrain the values between 0 and 1
        )

    def forward(self, vegetation_features):
        """
        vegetation_features: (B, T, vegetation_input_dim)
        """
        batch_size, time_steps, channels, height, width = vegetation_features.size()
        flattened_features = vegetation_features.view(time_steps * channels, height * width * batch_size, )

        # # Apply the gate on vegetation features

        gate_values = self.gate(flattened_features) # (B, T, 1)

        gated_vegetation_features = flattened_features * gate_values  # (B, T, vegetation_input_dim)

        gated_vegetation_features_reshaped = gated_vegetation_features.view(16, 1, 8, 81, 97)

        return gated_vegetation_features_reshaped


# Vegetation Processing model with gating mechanism
class VegetationProcessingWithGate(nn.Module):
    def __init__(self, input_dim, hidden_dims, kernel_size, num_layers, batch_first=True):
        super(VegetationProcessingWithGate, self).__init__()
        
        # Separate ConvLSTM to process vegetation features
        self.convlstm_vegetation = ConvLSTM(input_dim=input_dim,
                                            hidden_dim=hidden_dims,
                                            kernel_size=kernel_size,
                                            num_layers=num_layers,
                                            batch_first=batch_first,
                                            bias=True,
                                            return_all_layers=True)

        # Gating mechanism to control flow of information from vegetation features
        self.gating_mechanism = GatingMechanism(input_dim=input_dim, hidden_dim=8)  # Adjust hidden_dim as needed

        self.batch_norms = nn.ModuleList([nn.BatchNorm3d(hidden_dim) for hidden_dim in hidden_dims])

    def forward(self, vegetation_data):
        # Forward pass through the ConvLSTM for vegetation data

        layer_output_vegetation, _ = self.convlstm_vegetation(vegetation_data)

        for i, output in enumerate(layer_output_vegetation):
            output = output.permute(0, 2, 1, 3, 4)  # (B, C, T, H, W)
            output = self.batch_norms[i](output)
            output = output.permute(0, 2, 1, 3, 4)  # back to (B, T, C, H, W)
            layer_output_vegetation[i] = output

        # Apply gating mechanism to control the flow of vegetation data
        gated_vegetation_features = self.gating_mechanism(layer_output_vegetation[-1])  # Use last layer output

        return gated_vegetation_features  # Return gated features

# Combined Model with Gating Mechanism
class CombinedModelWithGate(nn.Module):
    def __init__(self, pretrained_model, vegetation_model, hidden_dims, kernel_size, num_layers, output_channels):
        super(CombinedModelWithGate, self).__init__()

        # Pre-trained ConvLSTM (Atmospheric Data) - freeze weights
        self.pretrained_model = pretrained_model
        for param in self.pretrained_model.parameters():
            param.requires_grad = False  # Freeze the pre-trained model

        # Model for Vegetation Data with Gating Mechanism
        self.vegetation_processing = vegetation_model

        # MLP for final regression/classification predictions
        self.mlp = MLP_5D2(height=81, width=97)

    def forward(self, x_atmospheric, x_vegetation):
        # Process atmospheric data through pre-trained ConvLSTM model
        _, _, atmospheric_features = self.pretrained_model(x_atmospheric)

        # Process vegetation data through the model with gating
        gated_vegetation_features = self.vegetation_processing(x_vegetation)

        # Combine the atmospheric and gated vegetation features
        combined_features = torch.cat([atmospheric_features, gated_vegetation_features], dim=2)

        # Final regression output through MLP
        regression_output = self.mlp(combined_features)

        return regression_output



In [ ]:
# Setting device
if torch.cuda.is_available():
    print("running on cuda")
    device = torch.device('cuda')
else:
    print("running on the cpu")
    device = torch.device('cpu')

In [ ]:
pretrained_model = ConvLSTMNetwork(
    input_dim=8 * set_lookback,
    hidden_dims=[8, 32, 64],
    kernel_size=(3, 3),
    num_layers=3,
    output_channels=8 * 1,
    batch_first=True
).to(device)

pretrained_model_path = "NEWPIPELINEConvLSTM_MULTI_TASK_NOTVEGREAL"

checkpoint = torch.load(pretrained_model_path, map_location=device)
pretrained_model.load_state_dict(checkpoint['model_state_dict'], strict=False)
pretrained_model.to(device)

vegetation_model = VegetationProcessingWithGate(input_dim=6, 
                                        hidden_dims=[8, 32, 8], 
                                        kernel_size=(3, 3), 
                                        num_layers=3).to(device)

combined_model = CombinedModelWithGate(pretrained_model, vegetation_model, 
                               hidden_dims=[8, 32, 64], kernel_size=(3, 3), 
                               num_layers=3, output_channels=8).to(device)




In [ ]:

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
vegetation_variables = [
    'UV visible albedo for direct radiation (climatological)',
    'Volumetric soil water layer 1',
    'Leaf area index, high vegetation',
    'Leaf area index, low vegetation',
    'Forecast surface roughness',
    'Evaporation'
]

# Definition of log epsilon for stability
epsilon = 1e-3

# Function to normalize using z-score
def z_score_normalisation(batch, mean, std, epsilon=1e-6):
    return (batch - mean) / (std + epsilon)

# Function to normalize using min-max scaling
def min_max_scaling(batch, min_value, max_value, epsilon=1e-6):
    return (batch - min_value) / (max_value - min_value + epsilon)

# Function to apply log transformation
def log_normalisation(batch, epsilon=1e-3):
    return torch.log(batch + epsilon)

def preprocess_data(data_loader, variable_names, global_stats, name):
    """
    Preprocess the entire dataset by normalizing the atmospheric and vegetation variables separately.
    """
    log_variables = ['Total precipitation', 'Volumetric soil water layer 1', 'Total column rain water']
    log_stats = {var: {'logs': []} for var in log_variables}  # Dictionary to store logs for each variable

    # First pass: Compute log mean and std for variables like 'Total precipitation'

    count = 0
    for X_batch, _ in tqdm(data_loader, desc="Computing log stats for variables"):

        if count == len(data_loader) -2:
            break

        count += 1
        
        for var in log_variables:
            var_idx = variable_names.index(var)
            X_var = X_batch[:, :, var_idx]  # Extract values for the variable
            if var == 'Total precipitation':
                X_var = X_var * 1000  # Convert precipitation to mm
            log_X_var = torch.log(X_var + epsilon)
            log_stats[var]['logs'].append(log_X_var.flatten().cpu().numpy())

    # Calculate the mean and std for log-transformed variables
    for var in log_variables:
        logs_all = np.concatenate(log_stats[var]['logs'])
        global_stats.setdefault(var, {})
        global_stats[var]['mean_log'] = np.mean(logs_all)
        global_stats[var]['std_log'] = np.std(logs_all)

    # Second pass: Normalize and separate atmospheric and vegetation data
    normalized_batches = []  # This will store the processed atmospheric data
    vegetation_data_batches = []  # This will store the vegetation data separately

    max_length = len(data_loader) - 2
    stop_count = 0

    for X_batch, y_batch in tqdm(data_loader, desc=f"Preprocessing {name.capitalize()} Data"):

        if stop_count == max_length:
            break 

        stop_count += 1
        
        X_batch = X_batch.clone()
        y_batch = y_batch.clone()

        new_X_batch = []  # Collect normalized atmospheric variables
        vegetation_features = []  # Collect vegetation features for later processing

        for var_idx, variable_name in enumerate(variable_names):
            X_var = X_batch[:, :, var_idx]  # Extract the variable

            # Process atmospheric variables 
            if variable_name not in vegetation_variables:
                # Atmospheric variables 
                if variable_name == 'Total precipitation':
                    X_var = X_var * 1000  # Convert to mm
                    mean_log = global_stats["Total precipitation"]['mean_log']
                    std_log = global_stats["Total precipitation"]['std_log']
                    X_var_normalized = z_score_normalisation(
                        log_normalisation(X_var, epsilon),
                        mean_log,
                        std_log
                    )
                
                # Other atmospheric variables
                elif variable_name in ['Volumetric soil water layer 1', 'Total column rain water']:
                    mean_log = global_stats[variable_name]['mean_log']
                    std_log = global_stats[variable_name]['std_log']
                    X_var_normalized = z_score_normalisation(
                        log_normalisation(X_var, epsilon),
                        mean_log,
                        std_log
                    )

                elif variable_name == 'Time-integrated surface latent heat net flux':
                    mean = global_stats[variable_name]['mean']
                    std = global_stats[variable_name]['std']
                    X_var_normalized = z_score_normalisation(X_var / 10800, mean, std)

                elif variable_name in ['2 metre temperature', '2 metre dewpoint temperature', 
                                    '10 metre U wind component', '10 metre V wind component']:
                    mean = global_stats[variable_name]['mean']
                    std = global_stats[variable_name]['std']
                    X_var_normalized = z_score_normalisation(X_var, mean, std)

                # Append the normalized atmospheric variable to new_X_batch
                new_X_batch.append(X_var_normalized.unsqueeze(2))

            # Collect the vegetation features separately (to be processed later)
            if variable_name in vegetation_variables:
                vegetation_features.append(X_var)  # Store vegetation features separately

        # Concatenate the normalized atmospheric variables
        final_X_batch = torch.cat(new_X_batch, dim=2)

        # Add zero indicator (for precipitation being zero)
        precip_idx = variable_names.index('Total precipitation')
        zero_indicator = (X_batch[:, :, precip_idx] == 0).float()
        final_X_batch = torch.cat((final_X_batch, zero_indicator.unsqueeze(2)), dim=2)

        # Convert y_batch to mm and store the zero indicator for precipitation
        y_zero_indicator = (y_batch == 0).float()
        y_batch = y_batch * 1000  # Convert target precipitation to mm

        # Store the final atmospheric data batches
        normalized_batches.append((final_X_batch, y_batch, y_zero_indicator))

        # Stack vegetation features for later processing (e.g., gating mechanism)
        vegetation_data_batches.append(torch.stack(vegetation_features, dim=2))  # Shape: (B, T, vegetation_input_dim)

    return normalized_batches, vegetation_data_batches


# # Precompute normalized data for atmospheric and vegetation variables separately
normalized_train_data, vegetation_train_data = preprocess_data(train_loader, variable_names, global_stats, "train")
normalized_val_data, vegetation_val_data = preprocess_data(val_loader, variable_names, global_stats, "val")
normalized_test_data, vegetation_test_data = preprocess_data(test_loader, variable_names, global_stats, "test")





In [ ]:
# Save the normalized data
torch.save(normalized_train_data, "normalized_train_data_TRANSFER_LEARNING_VEG.pth")
torch.save(normalized_val_data, "normalized_val_data_TRANSFER_LEARNING_VEG.pth")
torch.save(normalized_test_data, "normalized_test_data_TRANSFER_LEARNING_VEG.pth")

print("Saved normalized data successfully!")

In [ ]:
# Save the vegetation data in batches to avoid memory issues
torch.save(vegetation_train_data, "vegetation_train_data.pth")
torch.save(vegetation_val_data, "vegetation_val_data.pth")
torch.save(vegetation_test_data, "vegetation_test_data.pth")

# Confirm success
print("Vegetation data saved successfully.")


In [ ]:
#Definition of evaluation metrics
from scipy.stats import pearsonr, spearmanr
def nash_sutcliffe_efficiency(observed, predicted):
    # Ensure inputs are tensors on the CPU
    observed = observed.cpu()
    predicted = predicted.cpu()

    # Compute the numerator and denominator
    numerator = torch.sum((observed - predicted) ** 2)
    denominator = torch.sum((observed - torch.mean(observed)) ** 2)

    # Calculate NSE
    nse = 1 - (numerator / denominator)
    return nse.item()

from scipy.stats import pearsonr

def pearson_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return pearsonr(y_true, y_pred)[0]  # Return the correlation coefficient

from scipy.stats import spearmanr

def spearman_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return spearmanr(y_true, y_pred).correlation  # Return the Spearman correlation

def mse(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean((y_true - y_pred) ** 2).item()

def mae(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean(torch.abs(y_true - y_pred)).item()

def percentage_error(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return 100 * torch.mean((y_pred - y_true) / (y_true + 1e-6)).item()

def percentage_bias(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    return 100 * torch.sum(y_pred - y_true) / (torch.sum(y_true) + 1e-6)

import torch.nn.functional as F

def earth_movers_distance(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    # Compute EMD using the Wasserstein distance (L1 distance)
    emd = torch.mean(torch.abs(torch.sort(y_pred)[0] - torch.sort(y_true)[0])).item()
    return emd

def kendall_tau(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return kendalltau(y_true, y_pred).correlation  # Return the Kendall Tau

def r2_score(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
    ss_residual = torch.sum((y_true - y_pred) ** 2)
    
    return 1 - (ss_residual / (ss_total + 1e-6)).item()

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
# Load the normalized data
normalized_train_data = torch.load("normalized_train_data_TRANSFER_LEARNING_VEG.pth")
normalized_val_data = torch.load("normalized_val_data_TRANSFER_LEARNING_VEG.pth")
normalized_test_data = torch.load("normalized_test_data_TRANSFER_LEARNING_VEG.pth")

print("Loaded normalized data successfully!")


In [ ]:
# Load the preprocessed vegetation data from the saved .pth files
vegetation_train_data = torch.load("vegetation_train_data.pth")
vegetation_val_data = torch.load("vegetation_val_data.pth")
vegetation_test_data = torch.load("vegetation_test_data.pth")

# Confirm successful loading
print("Vegetation data loaded successfully.")


In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm  # Import tqdm for progress bars

# Load the preprocessed vegetation data
vegetation_train_data = torch.load("vegetation_train_data.pth")
vegetation_test_data = torch.load("vegetation_val_data.pth")


# Define optimizer and loss functions
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, combined_model.parameters()),  # Only train new layers
    lr=0.0005,
    weight_decay=0.01
)

loss_fn = nn.MSELoss()      
bce_loss_fn = nn.BCELoss() 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)


def train(model, train_loader, val_loader, num_epochs=100):
    model.train() 
    train_losses = []
    val_losses = []
    
    best_corr = -float('inf')  
    best_y_true = None
    best_y_pred = None
    best_y_true_corr = None
    best_y_pred_corr = None
    best_y_true_class = None
    best_y_pred_class = None

    for epoch in range(num_epochs):
        epoch_train_loss = 0.0
        epoch_val_loss = 0.0
        y_true = []
        y_pred = []

        for i, (X_batch, y_batch, _) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
            
            vegetation_features = vegetation_train_data[i].to(device) 
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            X_batch = X_batch.view(16, 1, 8, 81, 97)
            y_batch = y_batch.view(16, 1, 1, 81, 97)

            vegetation_features = vegetation_features.view(16, 1, 6, 81, 97)

            optimizer.zero_grad()

            regression_output = model(X_batch, vegetation_features)

            regression_loss = loss_fn(regression_output, y_batch)

            total_loss = regression_loss

            total_loss.backward()
            
            optimizer.step()

            epoch_train_loss += total_loss.item()

            spatial_corr_train = spatial_correlation(y_batch, regression_output)

            y_true.append(y_batch.cpu().numpy())
            y_pred.append(regression_output.cpu().detach().numpy())

        avg_train_loss = epoch_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()  # Set the model to evaluation mode
        epoch_val_loss = 0.0
        with torch.no_grad():  # No need to compute gradients during validation
            for i, (X_batch, y_batch, _) in tqdm(enumerate(val_loader), total=len(val_loader), desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):

                vegetation_features = vegetation_val_data[i].to(device)  # Use loaded vegetation data
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                
                X_batch = X_batch.view(16, 1, 8, 81, 97)
                y_batch = y_batch.view(16, 1, 1, 81, 97)

                vegetation_features = vegetation_features.view(16, 1, 6, 81, 97)

                # Forward pass
                regression_output = model(X_batch, vegetation_features)

                # Calculate the loss for regression
                regression_loss = loss_fn(regression_output, y_batch)

                # Total loss (combined)
                total_loss = regression_loss
                epoch_val_loss += total_loss.item()

                # Track predictions and true values for evaluation
                y_true_class = (y_batch > 0).float()  # Assume binary classification for precipitation
                y_pred_class = (regression_output > 0).float()
                
                # Calculate spatial correlation
                spatial_corr = spatial_correlation(y_batch, regression_output)

                # Save best correlation and loss
                if spatial_corr > best_corr:
                    best_corr = spatial_corr
                    best_y_true = y_batch.cpu().numpy()
                    best_y_pred = regression_output.cpu().detach().numpy()
                    best_y_true_corr = y_true_class.cpu().numpy()
                    best_y_pred_corr = y_pred_class.cpu().detach().numpy()
                    best_y_true_class = y_true_class.cpu().numpy()
                    best_y_pred_class = y_pred_class.cpu().detach().numpy()

        # Calculate average validation loss for this epoch
        avg_val_loss = epoch_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # Print progress and spatial correlation
        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Validation Loss: {avg_val_loss:.4f}, "
              f"Spatial Correlation: {best_corr:.4f}, "
              f"Spatial Correlation Train: {spatial_corr_train:.4f}")

        # Update the learning rate based on validation loss
        scheduler.step(avg_val_loss)

    # Save best validation results
    torch.save({
        'y_true_loss': best_y_true,
        'y_pred_loss': best_y_pred,
        'y_true_corr': best_y_true_corr,
        'y_pred_corr': best_y_pred_corr,
        'y_true_class': best_y_true_class,
        'y_pred_class': best_y_pred_class,
    }, 'best_validation_resultsMULTI_TASK_NOTVEGREAL_TRANSFERLEARNING.pth')

    print("Best validation predictions and ground truth saved.")

    return train_losses, val_losses

# Train the model
train_losses, val_losses = train(combined_model, normalized_train_data, normalized_val_data, num_epochs=100)



In [ ]:
# Save the model state dictionary
model_path = "NEWPIPELINEConvLSTM_MULTI_TASK_NOTVEGREAL_TRANSFERLEARNING"
torch.save({
    'model_state_dict': combined_model.state_dict(),
}, model_path)

In [ ]:
#Visualise the training and validation losses

import matplotlib.pyplot as plt

# Plot the training and validation loss for all of the folds combined
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epochs")
plt.ylabel("Training Dice Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.title("Validation Loss")
plt.legend()
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm  # Progress bar

# ---- 3. Average Saliency Map (for all features) ----
def compute_average_saliency_map(model, validation_data):
    """
    Computes the average saliency map across the entire validation dataset.
    """
    total_saliency = np.zeros((channels_in, height, width))
    total_samples = 0

    print("\nComputing average saliency map...")
    for batch_idx, (X_val, y_val, _) in enumerate(tqdm(validation_data, desc="Batches")):
        X_val = X_val.to(device)
        X_val = X_val.view(batch_size, time_steps_in, channels_in, height, width)

        for X_input in X_val:
            # Ensure proper shape and gradient computation
            X_input = X_input.unsqueeze(0).requires_grad_().to(device)
            model.zero_grad()

            # Forward pass
            regression_output, _ = model(X_input)
            loss = regression_output.sum()  # Sum all outputs to get gradients
            loss.backward()

            # Compute absolute gradients (saliency map)
            saliency = X_input.grad.abs().cpu().numpy()
            total_saliency += saliency[0, 0]  # Accumulate saliency maps for this sample
            total_samples += 1

    # Average the accumulated saliency maps
    average_saliency = total_saliency / total_samples

    # Plot saliency maps for all features
    for feature_idx in range(channels_in-1):
        plt.figure(figsize=(10, 6))
        plt.imshow(average_saliency[feature_idx], cmap='hot', interpolation='nearest')
        plt.title(f"Averaged Saliency Map for Feature {variable_names[feature_idx]} (Validation Set)")
        plt.colorbar()
        plt.show()

    return average_saliency


# ---- 4. Average Weights Inspection ----
def compute_average_weights(model):
    """
    Computes the average weight distributions across the entire model.
    """
    weights_dict = {}  # To store weights of all layers

    print("\nComputing average weights for all layers...")
    for name, param in tqdm(model.named_parameters(), desc="Layers"):
        if "weight" in name and param.requires_grad:
            weights = param.data.cpu().numpy().flatten()
            if name not in weights_dict:
                weights_dict[name] = []
            weights_dict[name].extend(weights)

    # Plot average weight distributions for each layer
    for layer, weights in weights_dict.items():
        plt.figure(figsize=(10, 6))
        plt.hist(weights, bins=50, color='blue', alpha=0.7)
        plt.title(f"Average Weight Distribution for Layer: {layer}")
        plt.xlabel("Weight Value")
        plt.ylabel("Frequency")
        plt.show()


# ---- Iterate Over Entire Validation Set ----
def evaluate_validation_batches(model, validation_data):
    """
    Evaluates the model by computing average saliency maps and average weight distributions.
    """
    print(f"\nEvaluating validation data with progress tracking...")

    # Compute average saliency maps
    compute_average_saliency_map(model, validation_data)

    # Compute average weights
    compute_average_weights(model)

    print("Completed evaluation for all validation batches.")


# ---- Run Evaluation ----
evaluate_validation_batches(model, normalized_val_data)


In [ ]:
#Definition of evaluation metrics
from scipy.stats import pearsonr, spearmanr
def nash_sutcliffe_efficiency(observed, predicted):
    # Ensure inputs are tensors on the CPU
    observed = observed.cpu()
    predicted = predicted.cpu()

    # Compute the numerator and denominator
    numerator = torch.sum((observed - predicted) ** 2)
    denominator = torch.sum((observed - torch.mean(observed)) ** 2)

    # Calculate NSE
    nse = 1 - (numerator / denominator)
    return nse.item()

from scipy.stats import pearsonr

def pearson_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return pearsonr(y_true, y_pred)[0]  # Return the correlation coefficient

from scipy.stats import spearmanr

def spearman_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return spearmanr(y_true, y_pred).correlation  # Return the Spearman correlation

def mse(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean((y_true - y_pred) ** 2).item()

def mae(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean(torch.abs(y_true - y_pred)).item()

def percentage_error(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return 100 * torch.mean((y_pred - y_true) / (y_true + 1e-6)).item()

def percentage_bias(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    return 100 * torch.sum(y_pred - y_true) / (torch.sum(y_true) + 1e-6)

import torch.nn.functional as F

def earth_movers_distance(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    # Compute EMD using the Wasserstein distance (L1 distance)
    emd = torch.mean(torch.abs(torch.sort(y_pred)[0] - torch.sort(y_true)[0])).item()
    return emd

def kendall_tau(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return kendalltau(y_true, y_pred).correlation  # Return the Kendall Tau

def r2_score(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
    ss_residual = torch.sum((y_true - y_pred) ** 2)
    
    return 1 - (ss_residual / (ss_total + 1e-6)).item()

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
pretrained_model = ConvLSTMNetwork(
    input_dim=8 * set_lookback,
    hidden_dims=[8, 32, 64],
    kernel_size=(3, 3),
    num_layers=3,
    output_channels=8 * 1,
    batch_first=True
).to(device)

pretrained_model_path = "NEWPIPELINEConvLSTM_MULTI_TASK_NOTVEGREAL"

checkpoint = torch.load(pretrained_model_path, map_location=device)
pretrained_model.load_state_dict(checkpoint['model_state_dict'], strict=False)
pretrained_model.to(device)

vegetation_model = VegetationProcessingWithGate(input_dim=6, 
                                        hidden_dims=[8, 32, 8], 
                                        kernel_size=(3, 3), 
                                        num_layers=3).to(device)

combined_model = CombinedModelWithGate(pretrained_model, vegetation_model, 
                               hidden_dims=[8, 32, 64], kernel_size=(3, 3), 
                               num_layers=3, output_channels=8).to(device)

checkpoint = torch.load("NEWPIPELINEConvLSTM_MULTI_TASK_NOTVEGREAL_TRANSFERLEARNING")
combined_model.load_state_dict(checkpoint['model_state_dict'])

combined_model.to(device)  

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, combined_model.parameters()), 
    lr=0.0005,
    weight_decay=0.01
)

loss_fn = nn.MSELoss()      
bce_loss_fn = nn.BCELoss()   
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

combined_model.eval()

print("Model loaded successfully")


In [ ]:

from scipy.stats import pearsonr, spearmanr
batch_size = batch_size
time_steps_out = 1
channels = 8
height = 81
width = 97
scaling_factor = 1

In [ ]:
def denormalize_precipitation(normalized_values, mean_log, std_log):
    """
    Denormalize precipitation data that was log-normalized and standardized.

    Parameters:
        normalized_values (torch.Tensor or np.ndarray): Normalized precipitation values.
        mean_log (float): Mean of log-transformed precipitation.
        std_log (float): Standard deviation of log-transformed precipitation.

    Returns:
        denormalized_values (np.ndarray): Denormalized precipitation values in mm.
    """
    # Reverse standardization
    log_values = (normalized_values * std_log) + mean_log

    # Reverse log transformation
    denormalized_values = np.exp(log_values) - epsilon

    # Convert back to mm (already done during normalization, so redundant here)
    return denormalized_values


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# Load saved predictions and ground truth
best_validation_predictions_and_ground_truth = torch.load('best_validation_resultsMULTI_TASK_NOTVEGREAL_TRANSFERLEARNING.pth')

# Regression outputs
y_true = best_validation_predictions_and_ground_truth['y_true_corr']
y_pred = best_validation_predictions_and_ground_truth['y_pred_corr']

# Ensure data is in numpy format if they are not already
if isinstance(y_true, np.ndarray):
    y_true = torch.from_numpy(y_true)  # Convert to tensor
if isinstance(y_pred, np.ndarray):
    y_pred = torch.from_numpy(y_pred)  # Convert to tensor

# Ensure data is in numpy format
y_predicted = y_pred
y_actual = y_true

# Performance metrics for regression
metrics = {
    "NSE": nash_sutcliffe_efficiency(y_actual, y_predicted),
    "R2": r2_score(y_actual, y_predicted),
    "Pearson": pearson_correlation(y_actual, y_predicted),
    "Spearman": spearman_correlation(y_actual, y_predicted),
    "MSE": mse(y_actual, y_predicted),
    "MAE": mae(y_actual, y_predicted),
    "Percentage Error": percentage_error(y_actual, y_predicted),
    "Percentage Bias": percentage_bias(y_actual, y_predicted),
    "EMD": earth_movers_distance(y_actual, y_predicted),
    "Kendall Tau": kendall_tau(y_actual, y_predicted),
    "Spatial Correlation": spatial_correlation(y_actual, y_predicted)
}

# Print regression metrics
print("\nRegression Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.16f}")

# Calculate mean and standard deviation of regression predictions
y_pred_array = np.array(y_pred.cpu())
y_true_array = np.array(y_true.cpu())

y_true_mean = np.mean(y_true_array)
y_pred_mean = np.mean(y_pred_array)

y_true_std = np.std(y_true_array)
y_pred_std = np.std(y_pred_array)

# Print mean and standard deviation
print("\nMean and SD of Ground Truth Precipitation:")
print(f"Mean: {y_true_mean:.16f}, SD: {y_true_std:.16f}")

print("\nMean and SD of Predicted Precipitation:")
print(f"Mean: {y_pred_mean:.16f}, SD: {y_pred_std:.16f}")



In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch 

best_validation_predictions_and_ground_truth = torch.load('best_validation_resultsMULTI_TASK_NOTVEGREAL.pth')

# Regression outputs
y_true = best_validation_predictions_and_ground_truth['y_true_loss']
y_pred = best_validation_predictions_and_ground_truth['y_pred_loss']

def plot_precipitation_distribution(precipitation_values, title):
    """
    Plot the histogram of precipitation values.
    """
    plt.figure(figsize=(10, 6))
    plt.hist(precipitation_values, bins=50, edgecolor='black', log=True)
    plt.title(f'Precipitation Distribution for {title}')
    plt.xlabel('Precipitation (mm)')
    plt.ylabel('Frequency (log scale)')
    plt.show()


def plot_scatter(y_true, y_pred, title):
    """
    Scatter plot of predicted vs ground truth precipitation.
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, s=10, c='blue')
    plt.xlabel("Ground Truth")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show()

def plot_spatial_heatmap(data, title):
    """
    Heatmap for spatial precipitation data.
    """
    mean_precip = np.mean(data, axis=0)  # Mean across time for spatial visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_precip, cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

# Flatten regression outputs for easier plotting
y_true_flat = y_true.cpu().numpy().flatten()
y_pred_flat = y_pred.cpu().numpy().flatten()

# Hexbin plots for regression
plt.figure(figsize=(8, 6))
hb = plt.hexbin(y_true_flat, y_pred_flat, gridsize=50, bins='log', cmap='plasma', mincnt=1)
plt.colorbar(hb, label='Count')
plt.xlabel("Ground Truth")
plt.ylabel("Predicted")
plt.title("Hexbin plot of ground truth vs predicted precipitation")
plt.show()

# Histograms for regression
plot_precipitation_distribution(y_true_flat, "Ground Truth Precipitation")
plot_precipitation_distribution(y_pred_flat, "Predicted Precipitation")

# Scatter plot for regression
plot_scatter(y_true_flat, y_pred_flat, "Scatter Plot of Ground Truth vs Predicted Precipitation")

# Heatmaps for spatial data
plot_spatial_heatmap(y_true[0, 0].cpu().numpy(), "Ground Truth Spatial Precipitation (First Timestamp)")
plot_spatial_heatmap(y_pred[0, 0].cpu().numpy(), "Predicted Spatial Precipitation (First Timestamp)")





In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, mean_absolute_error

threshold = 0.1
precip_index = 10

def evaluate(model, test_loader, loss_fn, device, height, width, vegetation_test_data):
    model.eval()  # Set the model to evaluation mode
    
    test_loss = 0.0
    y_true = []
    y_pred = []
    
    best_corr = -float('inf')
    best_y_true = None
    best_y_pred = None

    # Disable gradient computation
    with torch.no_grad():
        for X_test, y_test, _ in tqdm(test_loader, desc="Evaluating on Test Set"):
            # Get corresponding vegetation features for this batch
            vegetation_features = vegetation_test_data[0].to(device)  # Use loaded vegetation data

            X_test, y_test = X_test.to(device), y_test.to(device)

            # Reshape the inputs and targets if necessary
            batch_size = X_test.shape[0]
            X_test = X_test.view(batch_size, 1, 8, height, width)
            y_test = y_test.view(batch_size, 1, 1, height, width)

            vegetation_features = vegetation_features.view(batch_size, 1, 6, height, width)

            # Forward pass
            regression_output = model(X_test, vegetation_features)

            # Compute the regression loss
            reg_loss = loss_fn(regression_output, y_test)
            test_loss += reg_loss.item()

            # Collect true and predicted values
            y_true.append(y_test.cpu())
            y_pred.append(regression_output.cpu())

            # Compute spatial correlation
            correlation = spatial_correlation(y_test, regression_output)

            # Save the best results based on spatial correlation
            if correlation > best_corr:
                best_corr = correlation
                best_y_true = y_test
                best_y_pred = regression_output

    #Average loss over the entire test set
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}")

    #Calculate regression metrics
    y_true_flat = torch.cat(y_true, dim=0).flatten()
    y_pred_flat = torch.cat(y_pred, dim=0).flatten()

    mse = mean_squared_error(y_true_flat, y_pred_flat)
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    r2 = r2_score(y_true_flat, y_pred_flat)
    pearson_corr, _ = pearsonr(y_true_flat.numpy(), y_pred_flat.numpy())
    spearman_corr, _ = spearmanr(y_true_flat.numpy(), y_pred_flat.numpy())

    #Additional custom metrics
    nse = nash_sutcliffe_efficiency(y_true_flat, y_pred_flat)
    percentage_err = percentage_error(y_true_flat, y_pred_flat)
    percentage_bias_val = percentage_bias(y_true_flat, y_pred_flat)
    emd = earth_movers_distance(y_true_flat, y_pred_flat)
    kendall_corr = kendall_tau(y_true_flat, y_pred_flat)
    spatial_corr = spatial_correlation(y_true_flat, y_pred_flat)

    #Print regression metrics
    print("\nRegression Metrics:")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"NSE: {nse:.4f}")
    print(f"R2: {r2:.4f}")
    print(f"Pearson Correlation: {pearson_corr:.4f}")
    print(f"Spearman Correlation: {spearman_corr:.4f}")
    print(f"Percentage Error: {percentage_err:.4f}")
    print(f"Percentage Bias: {percentage_bias_val:.4f}")
    print(f"EMD: {emd:.4f}")
    print(f"Kendall Tau: {kendall_corr:.4f}")
    print(f"Spatial Correlation: {spatial_corr:.4f}")

    torch.save({
        'y_true_loss': best_y_true,
        'y_pred_loss': best_y_pred,
        'y_true_reg': torch.cat(y_true, dim=0).flatten(),
        'y_pred_reg': torch.cat(y_pred, dim=0).flatten()
    }, 'TESTING_RESULTS_VAL_TRANSFER_LEARNING.pth')

    print("Best validation predictions and ground truth saved.")

    return test_loss, mse, mae, r2, pearson_corr, spearman_corr, nse, percentage_err, percentage_bias_val, emd, kendall_corr, spatial_corr

normalized_val_data = torch.load("normalized_val_data_TRANSFER_LEARNING_VEG.pth")
vegetation_val_data = torch.load("vegetation_val_data.pth")

test_loss, mse, mae, r2, pearson_corr, spearman_corr, nse, percentage_err, percentage_bias_val, emd, kendall_corr, spatial_corr = evaluate(
    model=combined_model,
    test_loader=normalized_test_data,
    loss_fn=loss_fn,
    device=device,
    height=height,
    width=width,
    vegetation_test_data=vegetation_test_data
)

model_path = "best_trained_model_VAL_TRANSFERLEARNING.pth"
torch.save({
    'model_state_dict': combined_model.state_dict(),
}, model_path)
print(f"Model saved to {model_path}")

In [ ]:
results = torch.load('TESTING_RESULTS_VAL_TRANSFER_LEARNING.pth')

# Access the saved data
y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']
y_true_class = results['y_true_class']
y_pred_class = results['y_pred_class']
regression_metrics = {
    "MSE": mse(y_true_reg, y_pred_reg),
    "MAE": mae(y_true_reg, y_pred_reg),
    "NSE": nash_sutcliffe_efficiency(y_true_reg, y_pred_reg),
    "R2": r2_score(y_true_reg, y_pred_reg),
    "Pearson": pearson_correlation(y_true_reg, y_pred_reg),
    "Spearman": spearman_correlation(y_true_reg, y_pred_reg),
    "NSE": nash_sutcliffe_efficiency(y_true_reg, y_pred_reg),
    "Percentage Error": percentage_error(y_true_reg, y_pred_reg),
    "Percentage Bias": percentage_bias(y_true_reg, y_pred_reg),
    "EMD": earth_movers_distance(y_true_reg, y_pred_reg),
    "Kendall Tau": kendall_tau(y_true_reg, y_pred_reg),
    "Spatial Correlation": spatial_correlation(y_true_reg, y_pred_reg)}

print("\nRegression Metrics:")
for metric, value in regression_metrics.items():
    print(f"{metric}: {value:.16f}")


In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

def analyze_prediction_changes(y_true, y_pred_original, y_pred_vegetation):
    delta_predictions = np.abs(y_pred_original - y_pred_vegetation)
    
    print(f"Mean change: {np.mean(delta_predictions):.4f}")
    print(f"Standard deviation of change: {np.std(delta_predictions):.4f}")
    print(f"Max change: {np.max(delta_predictions):.4f}")
    print(f"Min change: {np.min(delta_predictions):.4f}")

    plt.figure(figsize=(8, 6))
    plt.hist(delta_predictions, bins=50, color='skyblue', edgecolor='black')
    plt.title('Distribution of Prediction Differences')
    plt.xlabel('Absolute Difference in Predictions')
    plt.ylabel('Frequency')
    plt.show()

    mae_original = np.mean(np.abs(y_true - y_pred_original))
    mae_vegetation = np.mean(np.abs(y_true - y_pred_vegetation))
    mse_original = np.mean((y_true - y_pred_original)**2)
    mse_vegetation = np.mean((y_true - y_pred_vegetation)**2)

    print(f"\nMAE (Original): {mae_original:.4f}")
    print(f"MAE (With Vegetation): {mae_vegetation:.4f}")
    print(f"MSE (Original): {mse_original:.4f}")
    print(f"MSE (With Vegetation): {mse_vegetation:.4f}")

    t_stat, p_value = stats.ttest_rel(y_pred_original, y_pred_vegetation)
    print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
    if p_value < 0.05:
        print("Significant difference between predictions with and without vegetation variables (p < 0.05)")
    else:
        print("No significant difference (p >= 0.05)")

results = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')

y_true_reg = results['y_true_reg']
y_pred_original = results['y_pred_reg']

results = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')

y_pred_veg = results['y_pred_reg']

# Example usage
analyze_prediction_changes(y_true_reg, y_pred_original, y_pred_reg)


In [ ]:
import numpy as np
import torch
import scipy.stats as stats
import matplotlib.pyplot as plt

results_original = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')
y_true_reg = results_original['y_true_reg'].cpu().numpy()
y_pred_original = results_original['y_pred_reg'].cpu().numpy()

results_veg = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')
y_pred_veg = results_veg['y_pred_reg'].cpu().numpy()

delta_y = np.abs(y_pred_original - y_pred_veg)

print(f"Mean change in predictions: {np.mean(delta_y):.4f}")
print(f"Standard deviation of change: {np.std(delta_y):.4f}")
print(f"Max change in predictions: {np.max(delta_y):.4f}")
print(f"Min change in predictions: {np.min(delta_y):.4f}")

plt.figure(figsize=(8, 6))
plt.hist(delta_y, bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of Prediction Differences')
plt.xlabel('Absolute Difference in Predictions')
plt.ylabel('Frequency')
plt.show()

mae_original = np.mean(np.abs(y_true_reg - y_pred_original))
mae_veg = np.mean(np.abs(y_true_reg - y_pred_veg))
mse_original = np.mean((y_true_reg - y_pred_original) ** 2)
mse_veg = np.mean((y_true_reg - y_pred_veg) ** 2)

print(f"\nMAE (Original): {mae_original:.4f}")
print(f"MAE (With Vegetation): {mae_veg:.4f}")
print(f"MSE (Original): {mse_original:.4f}")
print(f"MSE (With Vegetation): {mse_veg:.4f}")

bias_original = np.mean(y_pred_original - y_true_reg)
bias_veg = np.mean(y_pred_veg - y_true_reg)
print(f"\nBias (Original): {bias_original:.4f}")
print(f"Bias (With Vegetation): {bias_veg:.4f}")

pearson_original = stats.pearsonr(y_true_reg.flatten(), y_pred_original.flatten())[0]
pearson_veg = stats.pearsonr(y_true_reg.flatten(), y_pred_NOVEG.flatten())[0]
spearman_original = stats.spearmanr(y_true_reg.flatten(), y_pred_original.flatten())[0]
spearman_veg = stats.spearmanr(y_true_reg.flatten(), y_pred_NOVEG.flatten())[0]

print(f"\nPearson Correlation (Original): {pearson_original:.4f}")
print(f"Pearson Correlation (With Vegetation): {pearson_veg:.4f}")
print(f"Spearman Correlation (Original): {spearman_original:.4f}")
print(f"Spearman Correlation (With Vegetation): {spearman_veg:.4f}")

t_stat, p_value = stats.ttest_rel(y_pred_original.flatten(), y_pred_NOVEG.flatten())
print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("Significant difference between predictions with and without vegetation variables (p < 0.05)")
else:
    print("No significant difference (p >= 0.05)")

wilcoxon_stat, wilcoxon_p = stats.wilcoxon(y_pred_original.flatten(), y_pred_NOVEG.flatten(), zero_method='wilcox', mode='approx')
print(f"\nWilcoxon Test: statistic = {wilcoxon_stat:.4f}, p-value = {wilcoxon_p:.4f}")
if wilcoxon_p < 0.05:
    print("Significant difference (Wilcoxon test, p < 0.05)")
else:
    print("No significant difference (Wilcoxon test, p >= 0.05)")


In [ ]:
#Do the spatial plots 

results_original = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')
y_true_reg = results_original['y_true_reg'].cpu().numpy()  # Convert to numpy array
y_pred_original = results_original['y_pred_reg'].cpu().numpy()

results_veg = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')
y_pred_veg = results_veg['y_pred_reg'].cpu().numpy()

def plot_spatial_heatmap(y_true, y_pred_original, y_pred_veg, grid_shape, timestamp_idx=0):
    y_true_grid = y_true[timestamp_idx].reshape(grid_shape)
    y_pred_orig_grid = y_pred_original[timestamp_idx].reshape(grid_shape)
    y_pred_veg_grid = y_pred_veg[timestamp_idx].reshape(grid_shape)

    plt.figure(figsize=(15, 10))
    
    plt.subplot(1, 3, 1)
    plt.title(f"Ground Truth at Time {timestamp_idx}")
    sns.heatmap(y_true_grid, cmap="coolwarm", cbar=True)
    
    plt.subplot(1, 3, 2)
    plt.title(f"Original Model Prediction at Time {timestamp_idx}")
    sns.heatmap(y_pred_orig_grid, cmap="coolwarm", cbar=True)
    
    plt.subplot(1, 3, 3)
    plt.title(f"With Vegetation Prediction at Time {timestamp_idx}")
    sns.heatmap(y_pred_veg_grid, cmap="coolwarm", cbar=True)
    
    plt.show()

def plot_scatter(y_true, y_pred_original, y_pred_veg):
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true.flatten(), y_pred_original.flatten(), label="Original Model", alpha=0.5, color='blue')
    plt.scatter(y_true.flatten(), y_pred_NOVEG.flatten(), label="With Vegetation", alpha=0.5, color='red')
    plt.plot([0, np.max(y_true)], [0, np.max(y_true)], color='black', linestyle='--', label="Perfect Prediction")
    plt.xlabel("True Precipitation (mm)")
    plt.ylabel("Predicted Precipitation (mm)")
    plt.title("Scatter Plot: True vs Predicted Precipitation")
    plt.legend()
    plt.show()

plot_spatial_heatmap(y_true_reg, y_pred_original, y_pred_veg, 81*97, 0)

plot_scatter(y_true_reg, y_pred_original, y_pred_veg)


In [ ]:
#Error heatmaps to see if vegetation changes the spatial distribution of errors

results_original = torch.load('ConvLSTMtesting_results_MULTI_TASK.pth')
y_true_reg = results_original['y_true_reg'].cpu().numpy() 
y_pred_original = results_original['y_pred_reg'].cpu().numpy()

results_veg = torch.load('ConvLSTMtesting_results_MULTI_TASK_NOTVEGREAL.pth')
y_pred_veg = results_veg['y_pred_reg'].cpu().numpy()


def plot_error_heatmap(y_true, y_pred, grid_shape, title="Prediction Error"):
    error = np.abs(y_true - y_pred).reshape(grid_shape)
    plt.figure(figsize=(8, 6))
    sns.heatmap(error, cmap="Reds", cbar=True)
    plt.title(title)
    plt.show()

def plot_error_histogram(y_true, y_pred_original, y_pred_veg):
    error_original = y_true - y_pred_original
    error_veg = y_true - y_pred_veg

    plt.figure(figsize=(8, 6))
    plt.hist(error_original.flatten(), bins=50, alpha=0.5, label="Original Model", color='blue')
    plt.hist(error_NOVEG.flatten(), bins=50, alpha=0.5, label="With Vegetation", color='red')
    plt.title("Histogram of Prediction Errors")
    plt.xlabel("Prediction Error (mm)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

plot_error_heatmap(y_true_reg, y_pred_original, 81*97)

plot_error_histogram(y_true_reg, y_pred_original, y_pred_veg)


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

results = torch.load('TESTING_RESULTS_VAL_TRANSFER_LEARNING.pth')

y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']

def plot_precipitation_distribution(precipitation_values, title):
    """
    Plot the histogram of precipitation values.
    """
    plt.figure(figsize=(10, 6))
    plt.hist(precipitation_values, bins=50, edgecolor='black', log=True)
    plt.title(f'Precipitation Distribution for {title}')
    plt.xlabel('Precipitation (mm)')
    plt.ylabel('Frequency (log scale)')
    plt.show()

def plot_classification_rates(y_true_class, y_pred_class):
    """
    Plot the classification metrics: True Positive Rate, False Positive Rate, etc.
    """
    y_true_class = y_true_class.cpu().numpy() if isinstance(y_true_class, torch.Tensor) else y_true_class
    y_pred_class = y_pred_class.cpu().numpy() if isinstance(y_pred_class, torch.Tensor) else y_pred_class

    y_pred_binary = (y_pred_class > 0.5).astype(int)

    tp = np.sum((y_pred_binary == 1) & (y_true_class == 1))
    fp = np.sum((y_pred_binary == 1) & (y_true_class == 0))
    tn = np.sum((y_pred_binary == 0) & (y_true_class == 0))
    fn = np.sum((y_pred_binary == 0) & (y_true_class == 1))

    total = tp + fp + tn + fn
    rates = {
        "True Positive Rate": tp / total,
        "False Positive Rate": fp / total,
        "True Negative Rate": tn / total,
        "False Negative Rate": fn / total
    }

    plt.figure(figsize=(8, 6))
    sns.barplot(x=list(rates.keys()), y=list(rates.values()), palette="viridis")
    plt.title("Classification Rates")
    plt.ylabel("Rate")
    plt.xlabel("Metric")
    plt.ylim(0, 1)
    plt.xticks(rotation=45)
    plt.show()

def plot_scatter(y_true, y_pred, title):
    """
    Scatter plot of predicted vs ground truth precipitation.
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, s=10, c='blue')
    plt.xlabel("Ground Truth")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show()

def plot_spatial_heatmap(data, title, height=81, width=97):
    """
    Heatmap for spatial precipitation data.
    Assumes the data is provided as a flat array (1D) of size [time_steps * height * width].
    """
    num_time_steps = data.size // (height * width)
    data_reshaped = data.reshape((num_time_steps, height, width))

    # Compute mean precipitation across all time steps
    mean_precip = np.mean(data_reshaped, axis=0)

    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_precip, cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

plt.figure(figsize=(8, 6))
hb = plt.hexbin(y_true_reg, y_pred_reg, gridsize=50, bins='log', cmap='plasma', mincnt=1)
plt.colorbar(hb, label='Count')
plt.xlabel("Ground Truth")
plt.ylabel("Predicted")
plt.title("Hexbin Plot of Ground Truth vs Predicted Precipitation")
plt.show()

plot_precipitation_distribution(y_true_reg, "Ground Truth Precipitation")
plot_precipitation_distribution(y_pred_reg, "Predicted Precipitation")

plot_scatter(y_true_reg, y_pred_reg, "Scatter Plot of Ground Truth vs Predicted Precipitation")

plot_classification_rates(y_true_class, y_pred_class)

# # Spatial heatmaps for a few predictions (e.g., first 10 grids)
# num_visualizations = 10  # Change this to visualize more grids
# for i in range(num_visualizations):
#     start_idx = i * (81 * 97)
#     end_idx = start_idx + (81 * 97)

#     # Extract the specific grid
#     y_true_grid = y_true_reg[start_idx:end_idx].reshape(81, 97)
#     y_pred_grid = y_pred_reg[start_idx:end_idx].reshape(81, 97)
#     y_true_class_grid = y_true_class[start_idx:end_idx].reshape(81, 97)
#     y_pred_class_grid = y_pred_class[start_idx:end_idx].reshape(81, 97)

#     # Plot heatmaps
#     plot_spatial_heatmap(y_true_grid, f"Ground Truth Precipitation (Grid {i+1})")
#     plot_spatial_heatmap(y_pred_grid, f"Predicted Precipitation (Grid {i+1})")
#     plot_spatial_heatmap(y_true_class_grid, f"Ground Truth Zero Classification (Grid {i+1})")
#     plot_spatial_heatmap(y_pred_class_grid, f"Predicted Zero Classification (Grid {i+1})")




In [ ]:
results = torch.load('TESTING_RESULTS_VAL_TRANSFER_LEARNING.pth')

y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_predictions_top_row(y_true_reg, y_pred_reg, batch_index, num_samples, height=81, width=97, batch_size=16):
    """
    Visualize only the top row of predictions: Ground Truth, Predicted, and Residuals.
    If num_samples exceeds batch_size, it automatically moves to the next batch.

    - y_true_reg: Flattened ground truth tensor for regression.
    - y_pred_reg: Flattened predicted tensor for regression.
    - batch_index: Starting batch index to visualize.
    - num_samples: Total number of samples to visualize (can be > batch_size).
    - height: Height of the grid.
    - width: Width of the grid.
    - batch_size: Number of samples per batch.
    """

    samples_per_batch = height * width * batch_size
    total_samples = y_true_reg.shape[0] // (height * width) 
    num_samples = min(num_samples, total_samples)

    sample_count = 0  
    while sample_count < num_samples:
        # Compute batch-specific indices
        current_batch_index = batch_index + (sample_count // batch_size)
        start_idx = current_batch_index * samples_per_batch
        end_idx = start_idx + samples_per_batch

        # Ensure we don't go out of bounds
        if start_idx >= len(y_true_reg):
            print("Reached end of available batches.")
            break

        true_batch_reg = y_true_reg[start_idx:end_idx].reshape(batch_size, height, width)
        pred_batch_reg = y_pred_reg[start_idx:end_idx].reshape(batch_size, height, width) / 4

        residuals = true_batch_reg - pred_batch_reg
        remaining_samples = num_samples - sample_count
        samples_to_plot = min(remaining_samples, batch_size)

        for i in range(samples_to_plot):
            fig, axes = plt.subplots(1, 3, figsize=(18, 6)) 

            im1 = axes[0].imshow(true_batch_reg[i], cmap='viridis')
            axes[0].set_title(f"Batch {current_batch_index}, Sample {i} - Ground Truth (Regression)")
            plt.colorbar(im1, ax=axes[0])

            im2 = axes[1].imshow(pred_batch_reg[i], cmap='viridis')
            axes[1].set_title(f"Batch {current_batch_index}, Sample {i} - Predicted (Regression)")
            plt.colorbar(im2, ax=axes[1])

            residuals_np = residuals[i]
            im3 = axes[2].imshow(
                residuals_np,
                cmap='RdBu',
                vmin=-np.abs(residuals_np).max(),
                vmax=np.abs(residuals_np).max()
            )
            axes[2].set_title(f"Batch {current_batch_index}, Sample {i} - Residuals (True - Pred)")
            plt.colorbar(im3, ax=axes[2])

            plt.tight_layout()
            plt.show()

            sample_count += 1

plot_predictions_top_row(y_true_reg, y_pred_reg, batch_index=0, num_samples=32)




In [ ]:
def plot_predictions(y_true_reg, y_pred_reg, y_true_class, y_pred_class, batch_index, num_samples, threshold=0.5, height=81, width=97, batch_size=16):
    """
    Visualize the predictions, residuals, and classification for a given batch and selected samples.

    - y_true_reg: Flattened ground truth tensor for regression.
    - y_pred_reg: Flattened predicted tensor for regression.
    - y_true_class: Flattened ground truth tensor for classification.
    - y_pred_class: Flattened predicted tensor for classification.
    - batch_index: Index of the batch to visualize.
    - num_samples: Number of samples from the batch to visualize.
    - threshold: Classification threshold for visualization.
    - height: Height of the grid.
    - width: Width of the grid.
    - batch_size: Number of samples per batch.
    """

    import matplotlib.pyplot as plt
    import numpy as np

    samples_per_batch = height * width * batch_size

    start_idx = batch_index * samples_per_batch
    end_idx = start_idx + samples_per_batch

    true_batch_reg = y_true_reg[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_reg = y_pred_reg[start_idx:end_idx].reshape(batch_size, height, width)
    true_batch_class = y_true_class[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_class = y_pred_class[start_idx:end_idx].reshape(batch_size, height, width)

    pred_batch_class_thresholded = (pred_batch_class > threshold).float()

    residuals = true_batch_reg - pred_batch_reg

    for i in range(min(num_samples, batch_size)):
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        im1 = axes[0, 0].imshow(true_batch_reg[i].cpu().numpy(), cmap='viridis')
        axes[0, 0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Regression)")
        plt.colorbar(im1, ax=axes[0, 0])

        im2 = axes[0, 1].imshow(pred_batch_reg[i].cpu().numpy(), cmap='viridis')
        axes[0, 1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Regression)")
        plt.colorbar(im2, ax=axes[0, 1])

        residuals_np = residuals[i].cpu().numpy()
        im3 = axes[0, 2].imshow(
            residuals_np,
            cmap='RdBu',
            vmin=-np.abs(residuals_np).max(),
            vmax=np.abs(residuals_np).max()
        )
        axes[0, 2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (Regression)")
        plt.colorbar(im3, ax=axes[0, 2])

        im4 = axes[1, 0].imshow(true_batch_class[i].cpu().numpy(), cmap='binary')
        axes[1, 0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Classification)")
        plt.colorbar(im4, ax=axes[1, 0])

        im5 = axes[1, 1].imshow(pred_batch_class_thresholded[i].cpu().numpy(), cmap='binary')
        axes[1, 1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Classification, Thresholded)")
        plt.colorbar(im5, ax=axes[1, 1])

        class_residuals = true_batch_class[i] - pred_batch_class_thresholded[i]
        class_residuals_np = class_residuals.cpu().numpy()
        im6 = axes[1, 2].imshow(
            class_residuals_np,
            cmap='RdBu',
            vmin=-np.abs(class_residuals_np).max(),
            vmax=np.abs(class_residuals_np).max()
        )
        axes[1, 2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (Classification, Thresholded)")
        plt.colorbar(im6, ax=axes[1, 2])

        plt.tight_layout()
        plt.show()

plot_predictions(y_true_reg, y_pred_reg, y_true_class, y_pred_class, batch_index=0, num_samples=5, threshold=0.5)

